# Oil and Gas Anomaly Detection — Section 1b: Dataset Preparation (Class 9)

This notebook regenerates the **class 0 / class 9** dataset `final_data0_9.csv` with the same feature-engineering functions as notebook `01a`, but with a different selection of input files:

* **Class 0:** the first **12** Parquet files of folder `0` (instead of 30 in `01a`).
* **Class 9:** 14 explicitly selected Parquet files of folder `9` (list indices 2–7 and 151, 153, 154, 155, 157, 158, 159, 160).

The resulting class-9 table has 21,840 windows, which matches the class-9 count reported in Experiments 1 and 2, so this is the class-9 file used by the experiments.

*Documentation (markdown cells and `#` comment lines) was added for publication; the original executable code and its outputs are unchanged.*

## 1. Environment setup

In [2]:
# ============================================================
# 1. ENVIRONMENT SETUP
# ============================================================
# Import data-handling libraries and silence RuntimeWarnings.
import pandas as pd 
import numpy as np
import os
import glob
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

## 2. Feature engineering: statistical features per window

Identical to notebook `01a`: 16 statistics × 8 variables = 128 features per 30-second window.

In [3]:
# ============================================================
# 2. FEATURE ENGINEERING - STATISTICAL FEATURE EXTRACTION
# ============================================================
# Compute 16 whole-window statistical features for each of the 8 process
# variables of one window (group). `diffs` is computed but not used.
def extract_features(group: pd.DataFrame) -> pd.Series:
    feats = {}
    group = group.copy()
    variables = ['P-PDG',
        'P-TPT',
        'T-TPT',
        'P-MON-CKP',
        'T-JUS-CKP',
        'P-JUS-CKGL',
        'T-MON-CKP',
        'QGL'
    ]

    # Whole-sequence statistical features
    for col in variables:
        data = group[col]
        diffs = data.diff().fillna(0)

        # Basic stats
        feats[f'{col}_mean']       = data.mean()
        feats[f'{col}_std']        = data.std()
        feats[f'{col}_min']        = data.min()
        feats[f'{col}_max']        = data.max()
        feats[f'{col}_median']     = data.median()
        feats[f'{col}_range']      = data.max() - data.min()
        feats[f'{col}_q25']        = data.quantile(0.25)
        feats[f'{col}_q75']        = data.quantile(0.75)
        feats[f'{col}_iqr']        = data.quantile(0.75) - data.quantile(0.25)
        feats[f'{col}_mad']        = (data - data.mean()).abs().mean() # mean absolute deviation
        feats[f'{col}_mean_abs']   = data.abs().mean()
        feats[f'{col}_rms']        = np.sqrt((data**2).mean())      # root mean square
        feats[f'{col}_energy']     = (data**2).sum()                # signal energy

        # Optional: skewness & kurtosis (requires pandas >= 1.1 or scipy)
        feats[f'{col}_skew']       = data.skew()
        feats[f'{col}_kurtosis']   = data.kurtosis()

        # Zero‐crossing count (how many times signal changes sign)
        feats[f'{col}_zero_crossings'] = ((data.shift(1) * data) < 0).sum()
    return pd.Series(feats)

## 3. Data loading, labelling and window-based segmentation

Identical to notebook `01a`: transient-label mapping (`100 + k` → `k`), selection of the folder's class, 30-second non-overlapping windows, zero imputation, feature extraction and labelling.

In [4]:
# ============================================================
# 3. DATA LOADING, LABELLING AND 30-SECOND WINDOW SEGMENTATION
# ============================================================
# Process one Parquet file and return one row of features (+ label) per window.
def features_engineering(folder: str, filename: str) -> pd.DataFrame:
    
    # read
    path = os.path.join(folder, filename)
    data = pd.read_parquet(path, engine="pyarrow")

    data.reset_index(inplace = True )
    
    variables = [
        'P-PDG','P-TPT','T-TPT','P-MON-CKP','T-JUS-CKP','P-JUS-CKGL','T-MON-CKP','QGL'
    ]
    cols = ['timestamp'] + variables + ['class']
    data = data[cols].copy()
    
    # Fill NaN class values
    data['class'] = data['class'].dropna()

    # transient normalization (map 100+k -> k)
    transient_files = {1,2,5,6,7,8,9}
    
    num_of_transient = int(os.path.basename(folder))

    if num_of_transient in transient_files :
        
        mask = data['class'] == (100 + num_of_transient)
        data.loc[mask, 'class'] = num_of_transient
        
    # data_0 = data.loc[data['class'] == 0].copy()
        
    # # instance id: 1,2,3... every 30s from the first timestamp
    # t0 = data_0["timestamp"].min()
    # instance_numbers_0 = ((data_0["timestamp"] - t0).dt.total_seconds() // 30).astype(int) + 1
    # data_0["instance"] =  "class_0_" + instance_numbers_0.astype(str)

  
    data_t = data.loc[data['class'] == num_of_transient].copy()
    
    # instance id: 1,2,3... every 30s from the first timestamp
    t1 = data_t['timestamp'].min()
    instance_numbers_t = ((data_t['timestamp'] - t1).dt.total_seconds() // 30).astype('Int64') + 1
    data_t['instance'] = f"class_{num_of_transient}_" + instance_numbers_t.astype(str)

    data  = data_t.copy()

    data.drop(columns = ['timestamp'],inplace = True)
    data[variables]=data[variables].fillna(0)
    feature_data = data.groupby('instance').apply(extract_features,include_groups=False).fillna(0).reset_index()
    # 2. Add label per sequence
    label_data = data.groupby('instance')['class'].first().reset_index()
    feature_data = feature_data.merge(label_data, on='instance', how='left')
    return feature_data

# Folder 0

In [5]:
# ============================================================
# 4.0 CLASS 0 - NORMAL OPERATION
# ============================================================
# List the Parquet files of folder 0 (normal operation).
folder = '/kaggle/input/3w-dataset/2.0.0/0'
file_names = os.listdir(folder)

In [6]:
# Extract windowed statistical features from the first 12 files and concatenate them.
rows = []
for i  in file_names[:12]: 
    data = features_engineering(folder,i)
    rows.append(data)
    
final_data0= pd.concat(rows)

In [7]:
# Display the class-0 feature table with a fresh index (display only; not assigned).
final_data0.reset_index(drop= True)

,instance,P-PDG_mean,P-PDG_std,P-PDG_min,P-PDG_max,P-PDG_median,P-PDG_range,P-PDG_q25,P-PDG_q75,P-PDG_iqr,...,QGL_q75,QGL_iqr,QGL_mad,QGL_mean_abs,QGL_rms,QGL_energy,QGL_skew,QGL_kurtosis,QGL_zero_crossings,class
0,class_0_1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
1,class_0_10,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
2,class_0_100,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
3,class_0_101,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
4,class_0_102,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6887,class_0_95,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
6888,class_0_96,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
6889,class_0_97,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
6890,class_0_98,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0


In [8]:
# Display the class-0 feature table.
final_data0

,instance,P-PDG_mean,P-PDG_std,P-PDG_min,P-PDG_max,P-PDG_median,P-PDG_range,P-PDG_q25,P-PDG_q75,P-PDG_iqr,...,QGL_q75,QGL_iqr,QGL_mad,QGL_mean_abs,QGL_rms,QGL_energy,QGL_skew,QGL_kurtosis,QGL_zero_crossings,class
0,class_0_1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
1,class_0_10,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
2,class_0_100,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
3,class_0_101,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
4,class_0_102,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
593,class_0_95,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
594,class_0_96,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
595,class_0_97,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
596,class_0_98,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0


# TESTE

In [27]:
# Exploratory check: load one raw class-9 Parquet file.
test = pd.read_parquet('/kaggle/input/3w-dataset/2.0.0/9/SIMULATED_00008.parquet')

In [28]:
# Display the raw file.
test

,ABER-CKGL,ABER-CKP,ESTADO-DHSV,ESTADO-M1,ESTADO-M2,ESTADO-PXO,ESTADO-SDV-GL,ESTADO-SDV-P,ESTADO-W1,ESTADO-W2,...,PT-P,P-TPT,QBS,QGL,T-JUS-CKP,T-MON-CKP,T-PDG,T-TPT,class,state
timestamp,,,,,,,,,,,,,,,,,,,,,
2018-09-29 04:06:41,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,9461994.57,NaN,2.606090,78.888544,NaN,NaN,93.823964,0,0
2018-09-29 04:06:42,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,9461995.57,NaN,2.606091,78.888540,NaN,NaN,93.823954,0,0
2018-09-29 04:06:43,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,9461996.00,NaN,2.606092,78.888540,NaN,NaN,93.823944,0,0
2018-09-29 04:06:44,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,9461996.57,NaN,2.606093,78.888540,NaN,NaN,93.823934,0,0
2018-09-29 04:06:45,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,9461997.00,NaN,2.606094,78.888540,NaN,NaN,93.823924,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2018-09-29 16:46:35,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,16785695.20,NaN,0.001649,25.823692,NaN,NaN,77.321962,9,0
2018-09-29 16:46:36,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,16784772.80,NaN,0.001649,25.824998,NaN,NaN,77.319598,9,0
2018-09-29 16:46:37,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,16783127.20,NaN,0.001649,25.826014,NaN,NaN,77.317042,9,0


# Folder 9 


In [22]:
# ============================================================
# 4.9 CLASS 9
# ============================================================
# List the Parquet files of folder 9.
folder = '/kaggle/input/3w-dataset/2.0.0/9'
file_names = os.listdir(folder)

In [18]:
#file_names 

In [29]:
# Extract windowed features from 14 selected class-9 files (one call per file).
data_1= features_engineering(folder,file_names[2])

In [31]:
data_2= features_engineering(folder,file_names[3])

In [34]:
data_3 =features_engineering(folder,file_names[4])

In [36]:
data_4= features_engineering(folder,file_names[5])

In [38]:
data_5= features_engineering(folder,file_names[6])

In [40]:
data_6= features_engineering(folder,file_names[7]) 

In [42]:
data_7 = features_engineering(folder,file_names[151])

In [46]:
data_8 = features_engineering(folder,file_names[153])

In [50]:
data_9 = features_engineering(folder,file_names[154])

In [52]:
data_10 = features_engineering(folder,file_names[155])

In [56]:
data_11 = features_engineering(folder,file_names[157])

In [58]:
data_12 = features_engineering(folder,file_names[158])

In [60]:
data_13 = features_engineering(folder,file_names[159])

In [62]:
data_14 = features_engineering(folder,file_names[160])

In [64]:
# Concatenate the 14 class-9 feature tables.
final_data9 = pd.concat([data_1,data_2,data_3,data_4,data_5,data_6,data_7,data_8,data_9,data_10,data_11,data_12,data_13,data_14]).reset_index(drop=True)

In [66]:
# final_data9

In [67]:
# Remove windows labelled as normal operation (class 0).
final_data9 = final_data9[final_data9['class']!=0]

In [68]:
# Reset the row index after filtering.
final_data9.reset_index(drop= True, inplace = True)

In [70]:
# Display the class-9 feature table.
final_data9

,instance,P-PDG_mean,P-PDG_std,P-PDG_min,P-PDG_max,P-PDG_median,P-PDG_range,P-PDG_q25,P-PDG_q75,P-PDG_iqr,...,QGL_q75,QGL_iqr,QGL_mad,QGL_mean_abs,QGL_rms,QGL_energy,QGL_skew,QGL_kurtosis,QGL_zero_crossings,class
0,class_9_1,1.953203e+07,5.021770,1.953203e+07,1.953204e+07,1.953203e+07,10.000000,1.953203e+07,1.953204e+07,10.000000,...,2.615401,0.000058,0.000030,2.615369,2.615369,205.204605,-0.617624,-0.915332,0.0,9
1,class_9_10,1.953190e+07,4.983399,1.953190e+07,1.953191e+07,1.953190e+07,10.000000,1.953190e+07,1.953191e+07,10.000000,...,2.612647,0.000148,0.000077,2.612573,2.612573,204.766118,0.008979,-1.200614,0.0,9
2,class_9_100,1.953076e+07,6.888758,1.953075e+07,1.953077e+07,1.953076e+07,20.000000,1.953075e+07,1.953076e+07,10.000000,...,2.591216,0.000128,0.000066,2.591152,2.591152,201.422081,-0.002107,-1.196530,0.0,9
3,class_9_1000,2.651888e+07,141148.713689,2.629872e+07,2.675071e+07,2.652286e+07,451992.600000,2.640761e+07,2.663624e+07,228631.250000,...,0.004946,0.000010,0.000005,0.004941,0.004941,0.000732,0.000000,0.000000,0.0,9
4,class_9_1001,2.701758e+07,159161.436032,2.676636e+07,2.727542e+07,2.702224e+07,509062.200000,2.687174e+07,2.714772e+07,275979.600000,...,0.004925,0.000010,0.000005,0.004920,0.004920,0.000726,0.000000,0.000000,0.0,9
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21835,class_9_95,1.715209e+07,1519.275871,1.714959e+07,1.715460e+07,1.715209e+07,5010.000000,1.715084e+07,1.715334e+07,2504.375000,...,4.365014,0.028755,0.014873,4.350611,4.350645,567.843367,-0.012862,-1.199806,0.0,9
21836,class_9_96,1.715726e+07,1515.598171,1.715477e+07,1.715976e+07,1.715726e+07,4990.000000,1.715601e+07,1.715851e+07,2495.000000,...,4.304973,0.029790,0.015409,4.290051,4.290088,552.145641,-0.012840,-1.199792,0.0,9
21837,class_9_97,1.716246e+07,1537.051439,1.715994e+07,1.716500e+07,1.716245e+07,5060.000000,1.716119e+07,1.716372e+07,2532.500000,...,4.242770,0.041188,0.020734,4.221550,4.221617,534.661449,-0.135992,-1.261894,0.0,9
21838,class_9_98,1.716770e+07,1526.832665,1.716517e+07,1.717016e+07,1.716771e+07,4992.499999,1.716645e+07,1.716898e+07,2526.874999,...,4.155836,0.043605,0.022554,4.133993,4.134075,512.717264,-0.013416,-1.199885,0.0,9


## 5. Dataset assembly and export

In [71]:
# ============================================================
# 5. DATASET ASSEMBLY AND EXPORT
# ============================================================
# Combine the class-0 windows with the class-9 windows.
final_data0_9= pd.concat([final_data0,final_data9]).reset_index(drop=True)

In [72]:
# Drop the window identifier: only features and labels are kept.
final_data0_9.drop(columns=['instance'], inplace=True)

In [73]:
# Display the combined dataset.
final_data0_9

,P-PDG_mean,P-PDG_std,P-PDG_min,P-PDG_max,P-PDG_median,P-PDG_range,P-PDG_q25,P-PDG_q75,P-PDG_iqr,P-PDG_mad,...,QGL_q75,QGL_iqr,QGL_mad,QGL_mean_abs,QGL_rms,QGL_energy,QGL_skew,QGL_kurtosis,QGL_zero_crossings,class
0,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0
1,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0
2,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0
3,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0
4,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000e+00,0.000000e+00,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
28727,1.715209e+07,1519.275871,1.714959e+07,1.715460e+07,1.715209e+07,5010.000000,1.715084e+07,1.715334e+07,2504.375000,1294.083333,...,4.365014,0.028755,0.014873,4.350611,4.350645,567.843367,-0.012862,-1.199806,0.0,9
28728,1.715726e+07,1515.598171,1.715477e+07,1.715976e+07,1.715726e+07,4990.000000,1.715601e+07,1.715851e+07,2495.000000,1291.333333,...,4.304973,0.029790,0.015409,4.290051,4.290088,552.145641,-0.012840,-1.199792,0.0,9
28729,1.716246e+07,1537.051439,1.715994e+07,1.716500e+07,1.716245e+07,5060.000000,1.716119e+07,1.716372e+07,2532.500000,1309.166667,...,4.242770,0.041188,0.020734,4.221550,4.221617,534.661449,-0.135992,-1.261894,0.0,9
28730,1.716770e+07,1526.832665,1.716517e+07,1.717016e+07,1.716771e+07,4992.499999,1.716645e+07,1.716898e+07,2526.874999,1303.166666,...,4.155836,0.043605,0.022554,4.133993,4.134075,512.717264,-0.013416,-1.199885,0.0,9


In [74]:
# Export the combined class-0 / class-9 dataset used by the experiments.
# Save to CSV
final_data0_9.to_csv('final_data0_9.csv', index=False)